In [4]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# ── LOAD ────────────────────────────────────────────────────────
df = pd.read_csv("alumni_cleaned.csv")

COLORS = {
    'Job - Corporate':  '#2E86AB',
    'Job - Government': '#A23B72',
    'Higher Studies':   '#F18F01',
    'Misc':             '#C73E1D',
    'Not Reachable':    '#8B8B8B',
    'Unknown':          '#CCCCCC',
}
OUTCOME_COLORS = {
    'Placed':          '#2E86AB',
    'Higher Studies':  '#F18F01',
    'Other':           '#C73E1D',
}

short = {
    'Computer Science And Engineering':                             'CSE',
    'Data Science And Artificial Intelligence':                     'DSAI',
    'Electrical Engineering':                                       'EE',
    'Mechanical Engineering':                                       'ME',
    'Mathematics And Computing':                                    'Math & Computing',
    'Materials Science And Metallurgical Engineering':              'MSME',
    'Electronics And Communication Engineering':                    'ECE',
    'Control And Instrumentation':                                  'C&I',
    'Electric Vehicle Technology':                                  'EV Tech',
    'Design And Manufacturing':                                     'D&M',
    'Bioengineering':                                               'Bioengg',
    'Chemistry':                                                    'Chemistry',
    'Physics':                                                      'Physics',
    'Thermal And Fluids Engineering':                               'Thermal',
    'Power Systems And Power Electronics':                          'Power Sys',
    'Mechatronics Engineering':                                     'Mechatronics',
    'Liberal Arts':                                                 'Lib. Arts',
}
df['disc_short'] = df['discipline'].map(short).fillna(df['discipline'])

# ── normalize locations ─────────────────────────────────────────
loc_map = {'Bangalore':'Bengaluru','Gurugram':'Gurgaon',
           'Noida, Up':'Noida','Bengaluru ':'Bengaluru','Gurgaon ':'Gurgaon'}
df['location'] = df['location'].replace(loc_map)

# ═══════════════════════════════════════════════════════════════
# DASHBOARD — 6 panels in one HTML file
# ═══════════════════════════════════════════════════════════════
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=(
        "Overall Outcome Distribution",
        "Outcome by Course Type",
        "Placement % by Discipline",
        "Top 12 Recruiting Companies",
        "Top 10 Job Locations",
        "Discipline × Outcome Heatmap",
    ),
    specs=[
        [{"type": "pie"},       {"type": "bar"}],
        [{"type": "bar"},       {"type": "bar"}],
        [{"type": "bar"},       {"type": "heatmap"}],
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.10,
)

# ── Panel 1: Donut ──────────────────────────────────────────────
status_c = df['status'].value_counts()
fig.add_trace(go.Pie(
    labels=status_c.index,
    values=status_c.values,
    hole=0.5,
    marker_colors=[COLORS.get(s, '#AAAAAA') for s in status_c.index],
    textinfo='percent+label',
    textfont_size=11,
    showlegend=False,
), row=1, col=1)

# ── Panel 2: Grouped bar — outcome by course ────────────────────
course_order = ['BTech','BTech (Hons)','MTech','MSc','PhD','IDD']
for outcome, color in OUTCOME_COLORS.items():
    counts = []
    for c in course_order:
        n = len(df[(df['course'] == c) & (df['outcome'] == outcome)])
        counts.append(n)
    fig.add_trace(go.Bar(
        name=outcome, x=course_order, y=counts,
        marker_color=color, showlegend=True,
        text=counts, textposition='outside',
    ), row=1, col=2)

# ── Panel 3: Stacked horizontal — placement % by discipline ─────
disc_out = (
    df.groupby(['disc_short','outcome'])
    .size().unstack(fill_value=0)
)
disc_out['total'] = disc_out.sum(axis=1)
disc_out = disc_out.nlargest(10, 'total').drop(columns='total')
disc_pct = disc_out.div(disc_out.sum(axis=1), axis=0) * 100
disc_pct = disc_pct.sort_values('Placed', ascending=True)

for outcome, color in OUTCOME_COLORS.items():
    if outcome in disc_pct.columns:
        fig.add_trace(go.Bar(
            name=outcome,
            y=disc_pct.index.tolist(),
            x=disc_pct[outcome].round(1).tolist(),
            orientation='h',
            marker_color=color,
            showlegend=False,
            text=disc_pct[outcome].round(0).astype(int).astype(str) + '%',
            textposition='inside',
        ), row=2, col=1)

# ── Panel 4: Lollipop → bar for Plotly (top companies) ──────────
placed = df[df['status'] == 'Job - Corporate'].copy()
placed['company'] = placed['company'].replace({'Icici Bank':'Icici','Lti Mind Tree':'Ltimindtree'})
top_co = placed['company'].value_counts().head(12)

fig.add_trace(go.Bar(
    x=top_co.values,
    y=top_co.index,
    orientation='h',
    marker_color='#2E86AB',
    marker_line_color='white',
    text=top_co.values,
    textposition='outside',
    showlegend=False,
), row=2, col=2)

# ── Panel 5: Top locations ───────────────────────────────────────
job_df = df[df['outcome'] == 'Placed'].copy()
top_loc = job_df['location'].value_counts().dropna().head(10)

fig.add_trace(go.Bar(
    x=top_loc.index,
    y=top_loc.values,
    marker_color='#48CAE4',
    text=top_loc.values,
    textposition='outside',
    showlegend=False,
), row=3, col=1)

# ── Panel 6: Heatmap ─────────────────────────────────────────────
heat = (
    df.groupby(['disc_short','outcome'])
    .size().unstack(fill_value=0)
)
heat = heat.loc[heat.sum(axis=1).sort_values(ascending=False).index]

fig.add_trace(go.Heatmap(
    z=heat.values,
    x=heat.columns.tolist(),
    y=heat.index.tolist(),
    colorscale='YlOrRd',
    text=heat.values,
    texttemplate="%{text}",
    showscale=True,
    showlegend=False,
), row=3, col=2)

# ── Layout ───────────────────────────────────────────────────────
fig.update_layout(
    title=dict(
        text="<b>IIT Bhilai Alumni Placement Dashboard — 2025</b>",
        font=dict(size=20),
        x=0.5,
    ),
    height=1200,
    barmode='stack',
    plot_bgcolor='white',
    paper_bgcolor='#F8F9FA',
    font=dict(family='Arial', size=11),
    legend=dict(
        orientation='h', x=0.5, xanchor='center',
        y=1.02, yanchor='bottom',
        bgcolor='rgba(0,0,0,0)',
    ),
)
fig.update_xaxes(showgrid=True, gridcolor='#EEEEEE')
fig.update_yaxes(showgrid=True, gridcolor='#EEEEEE')

fig.write_html("dashboard.html")
print("✅ Saved: dashboard.html  — open this in your browser!")
fig.show()

✅ Saved: dashboard.html  — open this in your browser!
